In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
torch.cuda.is_available()

False

In [3]:
# importing training/testing data
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

c:\Users\yoava\OneDrive\Documents\Personal Projects\brainTumorIDv2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\yoava\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\2


In [4]:
#declare transforms

data_transforms = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [5]:
train_data = datasets.ImageFolder(f"{path}\Training", transform=data_transforms)
test_data = datasets.ImageFolder(f"{path}\Testing", transform=data_transforms)

class_names = train_data.class_to_idx
print(class_names)

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [6]:
train_dataloader = DataLoader(train_data, 32, shuffle=True)
test_dataloader = DataLoader(test_data, 32, shuffle=False)

In [10]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.model_def import ConvNet

In [11]:
model_0 = ConvNet(input_features = 1, 
                  output_features = 4)

In [18]:
def train_step(model, 
                data_loader,
                loss_fn,
                optimizer,
                accuracy_fn):

    train_loss, train_acc = 0, 0
    
    for batch, (X, y) in enumerate(data_loader):
        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        acc = accuracy_fn(y, y_pred.argmax(dim=1))

        train_loss += loss
        train_acc += acc

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        if batch % 50 == 0:
            print(f"Looked at {batch*len(X)}/{len(data_loader.dataset)} samples")

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

    print(f"Train loss: {train_loss}, Train accuracy: {train_acc}")

In [19]:
def test_step(model,
              data_loader,
              loss_fn,
              accuracy_fn):

    test_loss, test_acc = 0,0
    with torch.inference_mode():
        model.eval()
        for batch, (X, y) in enumerate(data_loader):
            y_pred = model(X)

            loss = loss_fn(y_pred, y)
            test_loss += loss

            acc = accuracy_fn(y, y_pred.argmax(dim=1))
            test_acc += acc

            if batch % 10 == 0:
                print(f"Looked at {batch*len(X)}/{len(data_loader.dataset)} samples")

        test_loss /= len(data_loader)
        test_acc /= len(data_loader)


    print(f"Test loss: {test_loss}, Test accuracy: {test_acc}")

In [20]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(),
                            lr = 0.005)

In [21]:
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

In [22]:
epochs = 3

for epoch in range(epochs):
    train_step(model_0, 
               train_dataloader,
               loss_fn,
               optimizer,
               accuracy_fn)

    test_step(model_0,
              test_dataloader,
              loss_fn,
              accuracy_fn)

Looked at 0/5600 samples


KeyboardInterrupt: 

In [ ]:
torch.save(model_0.state_dict(), 'model_0.pt')